In [1]:
from process import merge_datasets, deduplicate_dataset, remove_overlap

In [2]:
import os
import pandas as pd

# 合并四个数据集的train/val/test，分别去重得到merged_train, merged_val, merged_test

data_dir = '../../PPI-site/data'
datasets = ['pepnn', 'bitenet', 'interpep', 'pepbind']
splits = ['train', 'val', 'test']

merged_results = {}

# 合并四个数据集的train/val/test，分别内部去重得到merged_train, merged_val, merged_test
for split in splits:
    print(f'\n处理 {split} 数据集...')
    dfs = []
    
    # 读取四个数据集的对应split文件
    for dataset_name in datasets:
        file_path = os.path.join(data_dir, f'{dataset_name}_{split}.csv')
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            dfs.append(df)
            print(f'  加载 {dataset_name}_{split}: {len(df)} 行')
        else:
            print(f'   {dataset_name}_{split}.csv 不存在')
    
    if dfs:
        combined = merge_datasets(dfs)
        deduplicated = deduplicate_dataset(combined)
        deduplicated.to_csv(os.path.join(data_dir, f'merged_{split}.csv'), index=False)
        merged_results[f'merged_{split}'] = deduplicated
    else:
        print(f'    没有找到任何 {split} 数据集')

print(f'\n{"="*50}')
print('最终结果:')
for key, df in merged_results.items():
    print(f'  {key}: {len(df)} 行')
print(f'{"="*50}')



处理 train 数据集...
  加载 pepnn_train: 2517 行
  加载 bitenet_train: 1040 行
  加载 interpep_train: 225 行
  加载 pepbind_train: 576 行

处理 val 数据集...
  加载 pepnn_val: 311 行
  加载 bitenet_val: 75 行
  加载 interpep_val: 26 行
  加载 pepbind_val: 64 行

处理 test 数据集...
  加载 pepnn_test: 92 行
  加载 bitenet_test: 125 行
  加载 interpep_test: 251 行
  加载 pepbind_test: 639 行

最终结果:
  merged_train: 3574 行
  merged_val: 469 行
  merged_test: 978 行


In [3]:
# 合并merged_train和 Pretrain/7642.csv 作为最终的预训练任务数据集
# 同时去除所有val和test中出现过的序列组合

combined_train_pretrain = merge_datasets([merged_results['merged_train'], pd.read_csv('../7642.csv')])

# 去除val和test中出现过的序列组合
pretrain = remove_overlap(combined_train_pretrain, 
                             exclude_datasets=[merged_results['merged_val'], merged_results['merged_test']])
pretrain = deduplicate_dataset(pretrain)

output_path = '../data/pretrain.csv'
pretrain.to_csv(output_path, index=False)
print(f'\n已保存至: {output_path}')
print(f'最终训练数据行数: {len(pretrain)} 行')



已保存至: ../data/pretrain.csv
最终训练数据行数: 9292 行


In [4]:
# 调用generate_pretrain_datasets.py生成四个预训练任务的数据集
# 直接导入模块，避免subprocess的环境问题
import importlib.util
import sys

original_argv = sys.argv.copy()

sys.argv = ['generate_pretrain_datasets.py']

spec = importlib.util.spec_from_file_location("generate_pretrain_datasets", "generate_pretrain_datasets.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
module.main()

sys.argv = original_argv


开始生成预训练数据集
输入文件: ../data/pretrain.csv
输出目录: ../data
掩码比例: 0.4
窗口大小: 30
步长: 1
随机种子: 42
加载源数据: ../data/pretrain.csv
   原始数据行数: 9292
   列名: ['pdb_id', 'prot_seq', 'pep_seq', 'label', 'weight']
生成BMP数据集 (掩码比例: 0.4)
   掩码策略: 只掩码结合位点(label=1)和肽链位置
   掩码方式: BERT式 (80%[MASK] + 10%随机氨基酸 + 10%保持不变)
   处理进度: 1000/9292
   处理进度: 2000/9292
   处理进度: 3000/9292
   处理进度: 4000/9292
   处理进度: 5000/9292
   处理进度: 6000/9292
   处理进度: 7000/9292
   处理进度: 8000/9292
   处理进度: 9000/9292
BMP数据集生成完成: 9292 个样本
生成Dist2NBS数据集
   处理进度: 1000/9292
   处理进度: 2000/9292
   处理进度: 3000/9292
   处理进度: 4000/9292
   处理进度: 5000/9292
   处理进度: 6000/9292
   处理进度: 7000/9292
   处理进度: 8000/9292
   处理进度: 9000/9292
Dist2NBS数据集生成完成: 9292 个样本
生成PPB数据集
PPB数据集生成完成: 9292 个样本
生成SWBindCount数据集 (窗口大小: 30, 步长: 1)
   处理进度: 1000/9292
   处理进度: 2000/9292
   处理进度: 3000/9292
   处理进度: 4000/9292
   处理进度: 5000/9292
   处理进度: 6000/9292
   处理进度: 7000/9292
   处理进度: 8000/9292
   处理进度: 9000/9292
SWBindCount数据集生成完成: 1849833 个样本
保存数据集到: ../data
   BMP.csv: 9292 个样本
  

In [8]:
# 整理微调数据集
merged_train = pd.read_csv("../../PPI-site/data/merged_train.csv")
merged_val = pd.read_csv("../../PPI-site/data/merged_val.csv")
merged_test = pd.read_csv("../../PPI-site/data/merged_test.csv")

train = remove_overlap(
    df=merged_train,
    exclude_datasets=[merged_val, merged_test]
)
val = remove_overlap(
    df=merged_val,
    exclude_datasets=[merged_test]
)
test = merged_test
train = deduplicate_dataset(train)
val = deduplicate_dataset(val)
test = deduplicate_dataset(test)

train.to_csv("../../PPI-site/data/ft_train.csv")
val.to_csv("../../PPI-site/data/ft_val.csv")
test.to_csv("../../PPI-site/data/ft_test.csv")
